In [2]:
# This cell installs all the required libraries for the OratoAI project.
# The '!' character allows us to run terminal commands directly in a Colab notebook.
# The '-q' flag stands for "quiet" and reduces the amount of output during installation.

print("Installing required libraries for OratoAI...")

!pip install -q moviepy
# Used for video file manipulation, such as reading video duration
# and splitting the video and audio streams. (Module: Video Preprocessing)

!pip install -q SpeechRecognition
# A wrapper for various speech recognition APIs. We use it to transcribe a small
# audio sample to perform the language check. (Module: Video Preprocessing)

!pip install -q langdetect
# Used to detect the language of the transcribed text from the audio sample,
# ensuring the presentation is in English. (Module: Video Preprocessing)

!pip install -q pydub
# A high-level audio library used to handle audio segments easily. It helps in
# exporting the audio chunk for language detection. (Module: Video Preprocessing)

# Note: SpeechRecognition might require 'PyAudio' if you were to use a microphone,
# but for file-based processing, it's not needed. Colab also needs some additional
# dependencies for audio processing which are usually pre-installed.

print("\nInstallation complete. You can now proceed with running the video processing script.")


Installing required libraries for OratoAI...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 65.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 53.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done

Installation complete. You can now proceed with running the video processing script.


In [5]:
import os
from moviepy.editor import VideoFileClip
import speech_recognition as sr
from langdetect import detect, LangDetectException
from pydub import AudioSegment

# --- Configuration ---
MAX_DURATION_SECONDS = 300  # 5 minutes
SUPPORTED_LANGUAGE = 'en'   # ISO 639-1 code for English
TEMP_AUDIO_CHUNK = "temp_chunk.wav" # Temporary file for language detection

def process_video(video_path):
    """
    Processes an input video file by performing checks and splitting it into
    audio and video components.

    Args:
        video_path (str): The full path to the video file.

    Returns:
        tuple: A tuple containing the paths to the output video and audio files,
               or (None, None) if processing fails.
    """
    print(f"--- Starting processing for: {video_path} ---")

    # --- 1. Check if file exists ---
    if not os.path.exists(video_path):
        print(f"Error: File not found at '{video_path}'")
        return None, None

    try:
        # Load the video clip using moviepy
        clip = VideoFileClip(video_path)

        # --- 2. Preprocessing Check: Video Duration ---
        if clip.duration > MAX_DURATION_SECONDS:
            print(f"Error: Video duration ({clip.duration:.2f}s) exceeds the maximum limit of {MAX_DURATION_SECONDS}s.")
            clip.close()
            return None, None
        print(f"Success: Video duration is {clip.duration:.2f}s (within the limit).")

        # --- 3. Preprocessing Check: Language Detection ---
        print("Performing language check...")
        audio_for_check = clip.audio
        if audio_for_check is None:
             print("Error: Video has no audio track.")
             clip.close()
             return None, None

        # Export a small chunk for faster processing
        audio_for_check.subclip(0, min(15, clip.duration)).write_audiofile(TEMP_AUDIO_CHUNK, verbose=False, logger=None)

        # Use SpeechRecognition to get text from the audio chunk
        recognizer = sr.Recognizer()
        with sr.AudioFile(TEMP_AUDIO_CHUNK) as source:
            audio_data = recognizer.record(source)

        try:
            # Recognize speech using Google Web Speech API
            transcribed_text = recognizer.recognize_google(audio_data)
            print(f"Transcribed sample: '{transcribed_text}'")

            # Detect the language of the transcribed text
            detected_lang = detect(transcribed_text)
            if detected_lang != SUPPORTED_LANGUAGE:
                print(f"Error: Detected language is '{detected_lang}', but only '{SUPPORTED_LANGUAGE}' is supported.")
                # Clean up and exit
                os.remove(TEMP_AUDIO_CHUNK)
                clip.close()
                return None, None
            print(f"Success: Language confirmed as '{detected_lang}'.")

        except sr.UnknownValueError:
            print("Warning: Google Speech Recognition could not understand the audio. Could be non-speech or an unsupported accent. Proceeding anyway.")
        except sr.RequestError as e:
            print(f"Error: Could not request results from Google Speech Recognition service; {e}. Skipping language check.")
        except LangDetectException:
            print("Warning: Could not detect language from the transcribed text. It might be too short. Proceeding anyway.")
        finally:
             # Clean up the temporary chunk file
            if os.path.exists(TEMP_AUDIO_CHUNK):
                os.remove(TEMP_AUDIO_CHUNK)


        # --- 4. Split into Audio and Video Components ---
        print("Splitting video and audio...")
        base_name = os.path.splitext(os.path.basename(video_path))[0]
        output_video_path = f"{base_name}_video_only.mp4"
        output_audio_path = f"{base_name}_audio_only.wav"

        # Write the video file without the audio track
        clip.write_videofile(output_video_path, audio=False, verbose=False, logger=None)
        print(f"Success: Video component saved to '{output_video_path}'")

        # Write the audio file
        clip.audio.write_audiofile(output_audio_path, verbose=False, logger=None)
        print(f"Success: Audio component saved to '{output_audio_path}'")

        print("--- Processing complete. ---")
        return output_video_path, output_audio_path

    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None, None
    finally:
        # Ensure the clip is closed if it was opened
        if 'clip' in locals() and clip.reader:
            clip.close()


if __name__ == '__main__':
    user_video_file = "/content/clip_03.mp4"
    if user_video_file != "/content/clip_03.mp4":
        print("="*60)
        print("Please update the 'user_video_file' variable in the script")
        print("to the actual path of your video file to run this example.")
        print("="*60)
    else:
        process_video(user_video_file)

--- Starting processing for: /content/clip_03.mp4 ---
Success: Video duration is 17.20s (within the limit).
Performing language check...
Transcribed sample: 'keep it quick I'm doing mine and under a minute'
Success: Language confirmed as 'en'.
Splitting video and audio...


  warnings.warn("Warning: in file %s, "%(self.filename)+



Success: Video component saved to 'clip_03_video_only.mp4'
Success: Audio component saved to 'clip_03_audio_only.wav'
--- Processing complete. ---


In [16]:
print("Installing NVIDIA NeMo Toolkit for ASR... (This may take a few minutes)")
!pip install -U nemo_toolkit["asr"]
print("\nInstallation complete.")


Installing NVIDIA NeMo Toolkit for ASR... (This may take a few minutes)

Installation complete.


In [12]:
import os
import torch
import nemo.collections.asr as nemo_asr
from IPython.display import Audio, display

# --- 1. Setup GPU/CPU ---
if torch.cuda.is_available():
    device = 'cuda'
    print("✅ GPU is available. Using CUDA.")
else:
    device = 'cpu'
    print("⚠️ GPU not available. Using CPU (this will be significantly slower).")

# --- 2. Load the Pre-trained Parakeet Model ---
# This cell should only be run ONCE per session.
# It will download the model from NVIDIA's NGC cloud the first time it's run,
# and then load it into memory.
try:
    print("\nDownloading and loading the Parakeet ASR model (this may take a few moments)...")
    model_name = "nvidia/parakeet-tdt-0.6b-v2"
    print(f"Model: {model_name}")

    asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v2")
    asr_model.to(device)
    print("\n✅ Model loaded successfully and is ready for transcription.")

except Exception as e:
    print(f"\nAn unexpected error occurred during model setup: {e}")
    # Set model to None if loading fails
    asr_model = None


✅ GPU is available. Using CUDA.

Model: nvidia/parakeet-tdt-0.6b-v2


parakeet-tdt-0.6b-v2.nemo:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

[NeMo I 2025-10-11 08:04:08 nemo_logging:393] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2025-10-11 08:04:08 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 40.0
    min_duration: 0.1
    text_field: answer
    batch_duration: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2025-10-11 08:04:08 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config :

[NeMo I 2025-10-11 08:04:08 nemo_logging:393] PADDING: 0
[NeMo I 2025-10-11 08:04:12 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2025-10-11 08:04:12 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo W 2025-10-11 08:04:13 nemo_logging:405] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: Driver supports cuda toolkit version 12.4, but the driver needs to support at least 12,6. Please update your cuda driver.


[NeMo I 2025-10-11 08:04:13 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo W 2025-10-11 08:04:13 nemo_logging:405] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: Driver supports cuda toolkit version 12.4, but the driver needs to support at least 12,6. Please update your cuda driver.


[NeMo I 2025-10-11 08:04:27 nemo_logging:393] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--parakeet-tdt-0.6b-v2/snapshots/4f7f0088738aa056a90bdacbd6a0e22672b0f206/parakeet-tdt-0.6b-v2.nemo.

✅ Model loaded successfully and is ready for transcription.


In [21]:
import os
import soundfile as sf
import numpy as np

# This cell assumes 'asr_model' has been loaded in a previous cell.
# Example: asr_model = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained("nvidia/parakeet-tdt-0.6b-v2")

def ensure_mono_audio(audio_path):
    """
    Checks if an audio file is mono. If it's stereo, it converts it
    and saves a new mono version, returning the path to the mono file.
    If it's already mono, it returns the original path.
    """
    signal, sample_rate = sf.read(audio_path)

    # Check if audio is stereo (2 channels)
    if signal.ndim > 1 and signal.shape[1] == 2:
        print(f"Audio '{os.path.basename(audio_path)}' is Stereo. Converting to Mono...")
        mono_signal = np.mean(signal, axis=1)

        # Create a new filename for the mono version
        base, ext = os.path.splitext(audio_path)
        mono_path = f"{base}_mono.wav"

        # Save the mono file
        sf.write(mono_path, mono_signal, sample_rate)
        print(f"Mono version saved to: {mono_path}")
        return mono_path

    # If not stereo, it's already mono
    print(f"Audio '{os.path.basename(audio_path)}' is already Mono.")
    return audio_path

# --- Main Execution ---
if 'asr_model' in locals() and asr_model is not None:

    # 1. Define the path to your ORIGINAL audio file
    original_audio_file = '/content/clip_03_audio_only.wav' # <--- YOUR FILE HERE

    if not os.path.exists(original_audio_file):
        print(f"❌ Error: Audio file not found at '{original_audio_file}'")
    else:
        try:
            # 2. Ensure the audio is mono, getting the correct path
            mono_audio_path = ensure_mono_audio(original_audio_file)

            # 3. Transcribe using the OFFICIAL, simple method with the guaranteed mono file
            print("\nStarting transcription to get timestamps...")
            output = asr_model.transcribe([mono_audio_path], return_hypotheses=True)

            # 4. Extract and display timestamps from the hypothesis object
            # FIX: The correct attribute is .timestamp, not .timestep
            if output and isinstance(output, list) and hasattr(output[0], 'timestamp'):
                word_timestamps = output[0].timestamp['word']
                segment_timestamps = output[0].timestamp['segment']

                print("\n--- Segment-Level Timestamps ---")
                for stamp in segment_timestamps:
                     # Some newer versions use 'label' for the text segment
                     segment_text = stamp.get('label', stamp.get('segment', ''))
                     print(f"[{stamp['start_offset']:>5.2f}s - {stamp['end_offset']:>5.2f}s]: {segment_text}")

                print("\n--- Word-Level Timestamps ---")
                for stamp in word_timestamps:
                    print(f"[{stamp['start_offset']:>5.2f}s - {stamp['end_offset']:>5.2f}s]: {stamp['word']}")

            else:
                print("❌ Transcription did not return the expected timestamp object.")
                print("   The output was:", output)

        except Exception as e:
            print(f"\nAn unexpected error occurred: {e}")
else:
    print("\n❌ ASR Model not loaded. Please successfully run the model loading cell first.")


Audio 'clip_03_audio_only.wav' is Stereo. Converting to Mono...
Mono version saved to: /content/clip_03_audio_only_mono.wav

Starting transcription to get timestamps...


Transcribing: 100%|██████████| 1/1 [00:00<00:00,  3.99it/s]


--- Segment-Level Timestamps ---
[ 2.00s -  9.00s]: Yeah, they're gone.
[11.00s - 15.00s]: They're gone.
[16.00s - 23.00s]: We killed them.
[29.00s - 43.00s]: They're dead.
[47.00s - 93.00s]: I'm trying to think of the last time I watched an 18-minute TED Talk.
[96.00s - 125.00s]: It's been years, literally years.
[129.00s - 162.00s]: So if you're giving a TED Talk, keep it quick.
[165.00s - 184.00s]: I'm doing mine in under a minute.
[188.00s - 206.00s]: I'm at 44 seconds right now.

--- Word-Level Timestamps ---
[ 2.00s -  4.00s]: Yeah,
[ 5.00s -  7.00s]: they're
[ 7.00s -  9.00s]: gone.
[11.00s - 13.00s]: They're
[13.00s - 15.00s]: gone.
[16.00s - 18.00s]: We
[18.00s - 21.00s]: killed
[21.00s - 23.00s]: them.
[29.00s - 35.00s]: They're
[35.00s - 43.00s]: dead.
[47.00s - 51.00s]: I'm
[51.00s - 53.00s]: trying
[53.00s - 54.00s]: to
[54.00s - 56.00s]: think
[56.00s - 58.00s]: of
[58.00s - 61.00s]: the
[61.00s - 63.00s]: last
[63.00s - 66.00s]: time
[66.00s - 69.00s]: I
[69.00s - 74.00